# The Saga Pattern — Distributed Transactions Without 2PC

## 🧠 Mental Model

> **A Saga is like booking a flight+hotel+car rental. Each booking is independent.
> If the car rental fails, you must CANCEL the flight and hotel (compensating actions).
> There's no magic "undo all three simultaneously" — you execute the undos explicitly,
> one by one. Each step is a local transaction; the saga is the orchestration.**

### WHY Saga Exists — The Distributed Transaction Problem

```
Two-Phase Commit (2PC) — the naive solution:
  Phase 1: Coordinator asks all services "are you ready?"
  Phase 2: If all say yes → commit; if any says no → rollback

PROBLEMS with 2PC in microservices:
  - Coordinator is a SPOF — if it crashes between phases, participants are locked
  - All participants must be BLOCKED waiting for the decision (seconds!)
  - No support in HTTP/REST — only works in XA-capable systems (JDBC, JTA)
  - Performance: holds locks across N services simultaneously

SAGA solves this by: LOCAL transactions + COMPENSATING transactions (undo)
  - No distributed locks
  - Services remain available
  - Compensation is explicit business logic (not a rollback)
```

### Two Saga Flavours

```
CHOREOGRAPHY (event-driven)         ORCHESTRATION (command-driven)
────────────────────────────        ────────────────────────────────
No central coordinator              Central saga orchestrator
Services react to events            Orchestrator sends commands
  orders-service publishes           orders-service → calls payment-service
  "order.created"                    payment-service → calls inventory-service
  payment-service subscribes and     inventory-service → calls shipping-service
  publishes "payment.completed"
  inventory-service subscribes...

✓ Loose coupling                    ✓ Clear flow — easy to reason about
✓ No SPOF orchestrator             ✓ Easier to track state
✗ Hard to trace full flow          ✗ Orchestrator is a potential SPOF
✗ Business logic spread across      ✗ Introduces coupling to orchestrator
  many services
```


---
## ShopFlow Checkout Saga

### The Problem Scenario

```
ShopFlow checkout: 4 services must all succeed, or ALL must be undone.

1. Reserve inventory    (inventory-service)
2. Process payment      (payment-service)
3. Create order record  (orders-service)
4. Trigger fulfillment  (fulfillment-service)

IF step 3 fails after steps 1 and 2 have succeeded:
  → Inventory is reserved but no order exists
  → Payment was taken but nothing to deliver
  → Customer was charged for nothing

Compensating transactions needed:
  Undo step 2: REFUND payment
  Undo step 1: RELEASE inventory reservation
  (Undo step 4 not needed — it never ran)
```

### The Happy Path vs Failure Paths

```
HAPPY PATH:
  Reserve → Payment → Order → Fulfillment → ✓ Done

PAYMENT FAILS:
  Reserve → Payment FAILED
  → Compensate: Release reservation
  → Return error to customer

ORDER CREATION FAILS:
  Reserve → Payment → Order FAILED
  → Compensate: Refund payment → Release reservation
  → Return error to customer

FULFILLMENT FAILS:
  Reserve → Payment → Order → Fulfillment FAILED
  → Compensate: Cancel order → Refund payment → Release reservation
  → May trigger manual review
```


In [ ]:
from __future__ import annotations
import uuid, time, random
from enum import Enum
from dataclasses import dataclass, field
from typing import Callable

class SagaStatus(Enum):
    PENDING     = "pending"
    COMPLETED   = "completed"
    COMPENSATING = "compensating"
    FAILED      = "failed"

@dataclass
class SagaStep:
    name:        str
    action:      Callable       # the forward action
    compensate:  Callable       # the undo action
    completed:   bool = False

@dataclass
class SagaContext:
    order_id:       str = field(default_factory=lambda: f"ORD-{uuid.uuid4().hex[:6].upper()}")
    customer_id:    str = "cust-001"
    amount:         float = 99.99
    product_id:     str = "prod-widget"
    quantity:       int = 2
    reservation_id: str | None = None
    payment_ref:    str | None = None
    status:         SagaStatus = SagaStatus.PENDING
    log:            list[str]  = field(default_factory=list)

    def record(self, msg: str):
        ts = time.strftime("%H:%M:%S")
        self.log.append(f"[{ts}] {msg}")
        print(f"  {msg}")


# ── Service stubs (simulate real microservice calls) ─────────────────────────

def reserve_inventory(ctx: SagaContext, fail: bool = False) -> bool:
    if fail:
        ctx.record(f"inventory-service: reservation FAILED (out of stock)")
        return False
    ctx.reservation_id = f"RES-{uuid.uuid4().hex[:6].upper()}"
    ctx.record(f"inventory-service: reserved {ctx.quantity}× {ctx.product_id} "
               f"(reservation={ctx.reservation_id})")
    return True

def release_inventory(ctx: SagaContext) -> bool:
    ctx.record(f"inventory-service: COMPENSATE released reservation {ctx.reservation_id}")
    ctx.reservation_id = None
    return True

def process_payment(ctx: SagaContext, fail: bool = False) -> bool:
    if fail:
        ctx.record(f"payment-service: payment FAILED (card declined)")
        return False
    ctx.payment_ref = f"PAY-{uuid.uuid4().hex[:6].upper()}"
    ctx.record(f"payment-service: charged ${ctx.amount:.2f} → ref={ctx.payment_ref}")
    return True

def refund_payment(ctx: SagaContext) -> bool:
    ctx.record(f"payment-service: COMPENSATE refunded ${ctx.amount:.2f} "
               f"(ref={ctx.payment_ref})")
    ctx.payment_ref = None
    return True

def create_order(ctx: SagaContext, fail: bool = False) -> bool:
    if fail:
        ctx.record(f"orders-service: order creation FAILED (DB timeout)")
        return False
    ctx.record(f"orders-service: created order {ctx.order_id}")
    return True

def cancel_order(ctx: SagaContext) -> bool:
    ctx.record(f"orders-service: COMPENSATE cancelled order {ctx.order_id}")
    return True

def trigger_fulfillment(ctx: SagaContext, fail: bool = False) -> bool:
    if fail:
        ctx.record(f"fulfillment-service: FAILED (warehouse system down)")
        return False
    ctx.record(f"fulfillment-service: dispatched {ctx.order_id}")
    return True

def cancel_fulfillment(ctx: SagaContext) -> bool:
    ctx.record(f"fulfillment-service: COMPENSATE cancelled fulfillment for {ctx.order_id}")
    return True


# ── Saga Orchestrator ─────────────────────────────────────────────────────────

class SagaOrchestrator:
    def __init__(self, steps: list[SagaStep]):
        self._steps = steps

    def execute(self, ctx: SagaContext) -> bool:
        completed_steps: list[SagaStep] = []
        ctx.record(f"SAGA START: {ctx.order_id} for {ctx.customer_id}")

        for step in self._steps:
            success = step.action()
            if success:
                step.completed = True
                completed_steps.append(step)
            else:
                ctx.record(f"STEP FAILED: {step.name} → starting COMPENSATION")
                ctx.status = SagaStatus.COMPENSATING
                # Compensate in REVERSE order (last completed first)
                for done_step in reversed(completed_steps):
                    done_step.compensate()
                ctx.status = SagaStatus.FAILED
                ctx.record(f"SAGA FAILED: all compensations complete")
                return False

        ctx.status = SagaStatus.COMPLETED
        ctx.record(f"SAGA COMPLETE ✓")
        return True


# ── Demo: run three scenarios ─────────────────────────────────────────────────

def run_saga(scenario: str, inv_fail=False, pay_fail=False, ord_fail=False, ful_fail=False):
    print(f"
{'='*55}")
    print(f"SCENARIO: {scenario}")
    print('='*55)
    ctx = SagaContext()
    steps = [
        SagaStep("reserve_inventory",
                 action=lambda: reserve_inventory(ctx, fail=inv_fail),
                 compensate=lambda: release_inventory(ctx)),
        SagaStep("process_payment",
                 action=lambda: process_payment(ctx, fail=pay_fail),
                 compensate=lambda: refund_payment(ctx)),
        SagaStep("create_order",
                 action=lambda: create_order(ctx, fail=ord_fail),
                 compensate=lambda: cancel_order(ctx)),
        SagaStep("trigger_fulfillment",
                 action=lambda: trigger_fulfillment(ctx, fail=ful_fail),
                 compensate=lambda: cancel_fulfillment(ctx)),
    ]
    orchestrator = SagaOrchestrator(steps)
    result = orchestrator.execute(ctx)
    print(f"
  Final status: {ctx.status.value.upper()} | Reservation: {ctx.reservation_id} | Payment: {ctx.payment_ref}")
    return result

run_saga("Happy Path — all steps succeed")
run_saga("Payment fails — inventory must be released", pay_fail=True)
run_saga("Order creation fails — payment refunded + inventory released", ord_fail=True)


---
## Saga vs 2PC: When to Choose What

| Criteria | 2PC | Saga |
|---|---|---|
| Data consistency | Strong (atomic) | Eventual (compensate-based) |
| Performance | Poor (locks held across all services) | Good (local transactions only) |
| Complexity | Simple (database handles it) | Complex (explicit compensation logic) |
| Failure tolerance | Coordinator SPOF | Each service independent |
| Use when | Same database, OLTP | Microservices, cross-service transactions |

### ⚠️ Saga Gotchas

**1. Compensation is NOT always possible (non-reversible actions)**
```
Sending an email is irreversible. Compensation = "send another email saying sorry."
Charging a credit card has a refund window (Stripe: 90 days).
Physical goods already shipped cannot be "un-shipped" instantly.
Design compensation as "best effort business action," not a technical undo.
```

**2. Isolation is lost (dirty reads between saga steps)**
```
Step 2 (payment) succeeds. Another service reads "payment processed."
Step 3 (order) fails. Payment gets refunded.
That other service now has stale state.
Fix: semantic locks, versioned state, "pending" states visible to others.
```

**3. Saga itself can fail mid-compensation**
```
If the orchestrator crashes during compensation, some undos may not run.
Fix: idempotent compensations + saga log stored in durable storage.
On restart, resume compensation from where it stopped.
```

**4. Choreography leads to "saga spaghetti"**
```
With 5 services each reacting to events from others, tracing a failed saga
across logs from 5 services is a debugging nightmare.
Fix: distributed tracing (Jaeger, Zipkin) + correlation IDs on every event.
```

### 🌍 Where Saga is Used in Production

| Company | Usage |
|---|---|
| Uber | Trip booking saga: pricing → payment → driver assignment → fulfillment |
| Netflix | Content publishing saga: transcode → encrypt → store → CDN invalidation |
| Amazon | Order saga: payment → inventory → warehouse → shipping |
| Shopify | Checkout saga: inventory reserve → payment → order create → fulfillment |
